In [ ]:
import requests
from bs4 import BeautifulSoup
import re

# 1. 선샌니가 보고 계신 PC 주소로 변경!
url = "https://cafe.naver.com/projectiofficial" 

# 2. 크롬 브라우저인 척 위장
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# 3. 데이터 요청
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')

try:
    #  멤버수 (캡처 화면을 바탕으로 제가 미리 짰습니다!)
    member_element = soup.select_one('.mem-cnt-info em') 
    member_text = member_element.text if member_element else "0"

    #  전체글보기 (선샌니가 Copy selector 한 값을 아래 따옴표 안에 붙여넣어주세요!)
    # 예시: post_element = soup.select_one('#cafe-info-data > ul > li:nth-child(4) > em')
    post_element = soup.select_one('#cafe-menu > div.box-g-m > ul:nth-child(1) > li:nth-child(1)') 
    post_text = post_element.text if post_element else "0"

    # 숫자만 깔끔하게 빼기
    member_count = int(re.sub(r'[^0-9]', '', member_text))
    total_posts = int(re.sub(r'[^0-9]', '', post_text))

    print(f" 카페 가입자 수: {member_count:,}명")
    print(f" 전체 게시글 수: {total_posts:,}개")

except Exception as e:
    print(f"데이터 추출 실패  오류 내용: {e}")

✅ 카페 가입자 수: 25,279명
✅ 전체 게시글 수: 127,955개


In [7]:
import requests
from bs4 import BeautifulSoup
import re

url = "https://cafe.naver.com/projectiofficial" 
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')

try:
    # 1. 가입자 수 (성공했던 완벽한 코드 그대로!)
    member_element = soup.select_one('.mem-cnt-info em')
    member_count = int(re.sub(r'[^0-9]', '', member_element.text)) if member_element else 0
    print(f"✅ 카페 가입자 수: {member_count:,}명")

    # 2. 게시글 수 (진짜 찐 불도저 방식 🚜)
    total_posts = 0
    
    # 🌟 핵심: 특정 구역을 찾지 않고, 페이지 내의 모든 <li> 태그를 싹 다 가져옵니다.
    all_li_tags = soup.find_all('li') 
    
    for li in all_li_tags:
        # 공백과 줄바꿈을 다 지우고 깔끔한 텍스트만 봅니다.
        text_content = li.text.strip().replace('\n', '').replace(' ', '')
        
        # 만약 그 줄에 '전체글'이라는 단어가 포함되어 있다면?!
        if '전체글' in text_content:
            # 글자는 무시하고 숫자(콤마 포함)만 모조리 긁어옵니다.
            numbers = re.findall(r'[\d,]+', text_content)
            
            if numbers:
                # 찾은 숫자 덩어리 중 첫 번째 값을 가져와서 콤마 떼고 숫자로 변환!
                total_posts = int(numbers[0].replace(',', ''))
                break # 찾았으니 반복문 즉시 종료!

    print(f"✅ 전체 게시글 수: {total_posts:,}개")

except Exception as e:
    print(f"오류 발생: {e}")

✅ 카페 가입자 수: 25,279명
✅ 전체 게시글 수: 127,957개


In [1]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd
from concurrent.futures import ThreadPoolExecutor # 병렬 처리를 위한 일꾼 소환!

# 1. 분석할 카페 아이디 리스트 (여기에 왕창 넣으셔도 됩니다!)
cafe_ids = ['projectiofficial', 'tteokbokk1', 'syoka?email=aba19787d0dde87bee8a21ff0ec27e86', 'dunggeure', 'akakang']

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# 2. 한 개의 카페를 수집하는 함수 정의 (선샌니가 성공시킨 그 불도저 로직!)
def scrape_cafe(cafe_id):
    url = f"https://cafe.naver.com/{cafe_id}"
    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # 가입자 수
        member_element = soup.select_one('.mem-cnt-info em')
        member_count = int(re.sub(r'[^0-9]', '', member_element.text)) if member_element else 0
        
        # 게시글 수 (불도저 방식)
        total_posts = 0
        all_li_tags = soup.find_all('li')
        for li in all_li_tags:
            text_content = li.text.strip().replace('\n', '').replace(' ', '')
            if '전체글' in text_content:
                numbers = re.findall(r'[\d,]+', text_content)
                if numbers:
                    total_posts = int(numbers[0].replace(',', ''))
                break
        
        print(f"✅ {cafe_id} 수집 완료!")
        return {'카페아이디': cafe_id, '가입자수': member_count, '게시글수': total_posts}
    
    except Exception as e:
        print(f"❌ {cafe_id} 실패: {e}")
        return {'카페아이디': cafe_id, '가입자수': 0, '게시글수': 0}

# 3. 멀티스레드 실행 (동시에 5개씩 드가자!)
print("🏁 멀티스레드 크롤링을 시작합니다...")
with ThreadPoolExecutor(max_workers=5) as executor:
    results = list(executor.map(scrape_cafe, cafe_ids))

# 4. 결과 저장
df = pd.DataFrame(results)
df.to_csv('cime_parallel_data.csv', index=False, encoding='utf-8-sig')

print(f"\n🎉 총 {len(df)}개 카페 수집 완료! 'cime_parallel_data.csv'를 확인해 보세요.")

🏁 멀티스레드 크롤링을 시작합니다...
✅ tteokbokk1 수집 완료!
✅ akakang 수집 완료!
✅ syoka?email=aba19787d0dde87bee8a21ff0ec27e86 수집 완료!
✅ dunggeure 수집 완료!
✅ projectiofficial 수집 완료!

🎉 총 5개 카페 수집 완료! 'cime_parallel_data.csv'를 확인해 보세요.


In [4]:
# 🔍 긴급 진단: 파이썬은 지금 무엇을 보고 있는가?
import requests
import urllib.parse # 👈 요 녀석이 없어서 에러가 났던 거예요!
import re
test_name = "스텔라이브" # 혹은 결과가 확실한 스트리머 이름
search_keyword = urllib.parse.quote(f"{test_name} 공식 팬카페")
url = f"https://m.search.naver.com/search.naver?where=m_cafe&query={search_keyword}"

res = requests.get(url, headers=headers)
print(f"📡 응답 상태: {res.status_code}")

if "로봇이 아닙니다" in res.text:
    print("❌ 경보: 네이버가 우리를 봇으로 감지했습니다! IP가 차단된 상태입니다.")
elif "카페" not in res.text:
    print("❌ 경보: 검색 결과에 카페 정보가 아예 없습니다. 페이지 구조가 바뀌었을 수 있습니다.")
else:
    print("✅ 일단 네이버 검색 결과는 잘 들어오고 있습니다!")

📡 응답 상태: 403
✅ 일단 네이버 검색 결과는 잘 들어오고 있습니다!


In [1]:
import os
import random
import time
import re
import urllib.parse
import pandas as pd
import requests

# 1. 파일 불러오기 (이름 다시 한번 확인!)
file_path = 'final_streamer_dataset_1p_122p.csv' 

try:
    df = pd.read_csv(file_path, encoding='utf-8-sig')
except:
    df = pd.read_csv(file_path, encoding='cp949')

unique_streamers = df['streamer_name'].dropna().unique().tolist()
target_streamers = unique_streamers[:500] # 딱 500명만!

print(f"📊 [200 OK 확인!] 1차 타겟 500명 수집을 다시 시작합니다!")

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
}

# 🌟 더 강력해진 '그물형' 수집 함수
def get_cafe_id_final(streamer_name):
    search_keyword = urllib.parse.quote(f"{streamer_name} 공식 팬카페") 
    url = f"https://m.search.naver.com/search.naver?where=m_cafe&query={search_keyword}"
    
    try:
        res = requests.get(url, headers=headers, timeout=10)
        if res.status_code != 200:
            return f"Error({res.status_code})"
        
        # 진단 코드에서 썼던 그 방식! 정규표현식으로 아이디 싹 긁기
        ids = re.findall(r'cafe\.naver\.com/([a-zA-Z0-9_]+)', res.text)
        
        # 중복 제거하고 쓸모없는 아이디(naver, search 등) 걸러내기
        candidates = []
        for cid in ids:
            if len(cid) > 3 and cid not in ['naver', 'search', 'my', 'section']:
                if cid not in candidates:
                    candidates.append(cid)
        
        # 첫 번째 후보를 바로 반환!
        return candidates[0] if candidates else None
    except:
        return None

# 3. 드가자~~!!
results = []
# ... 앞부분 동일 ...
for i, name in enumerate(target_streamers):
    cafe_id = get_cafe_id_final(name)
    
    # 403이 뜨면 즉시 중단하고 알려주기
    if cafe_id and "Error(403)" in str(cafe_id):
        print(f"🚨 [비상] {i+1}번째에서 다시 차단당했습니다! 핫스팟 IP를 재할당하거나 더 오래 쉬어야 해요.")
        break
        
    results.append({'streamer_name': name, 'cafe_id': cafe_id})
    print(f"🎯 [{i+1}/500] {name} ➡️ {cafe_id if cafe_id else '못 찾음 😭'}")
    
    # 지연 시간을 더 길고 불규칙하게! (3초 ~ 6초)
    time.sleep(random.uniform(3.0, 6.0))
# ... 뒷부분 동일 ...

# 최종 저장
pd.DataFrame(results).to_csv('mapping_test_500_final.csv', index=False, encoding='utf-8-sig')
print("\n🎊 선샌니! 500명 찍먹 완료! 'mapping_test_500_final.csv' 확인 드가자!")

📊 [200 OK 확인!] 1차 타겟 500명 수집을 다시 시작합니다!
🎯 [1/500] 00곤뇽00 ➡️ 못 찾음 😭
🎯 [2/500] 0리엘0 ➡️ bandjannabi
🎯 [3/500] 0무지0 ➡️ m2school
🎯 [4/500] 0서이안0 ➡️ 0nepunchk1ng
🎯 [5/500] 0영식이 ➡️ takstudioofficial
🎯 [6/500] 0우리아0 ➡️ khantata
🎯 [7/500] 1010m ➡️ 못 찾음 😭
🎯 [8/500] 1김상어1 ➡️ 못 찾음 😭
🎯 [9/500] 4대가호 최하엘 ➡️ 못 찾음 😭
🎯 [10/500] 8Fabe8 ➡️ 못 찾음 😭
🎯 [11/500] 9unD ➡️ 못 찾음 😭
🎯 [12/500] ADIA 아디아 ➡️ 못 찾음 😭
🎯 [13/500] B 059 비오 ➡️ 못 찾음 😭
🎯 [14/500] BJ쯔코 ➡️ clidfan
🎯 [15/500] BJ펭귄 ➡️ clidfan
🎯 [16/500] Bomnun ➡️ 못 찾음 😭
🎯 [17/500] breeeezy ➡️ 못 찾음 😭


KeyboardInterrupt: 

In [12]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import urllib.parse
import re
import time

# 1. 선샌니의 데이터 프레임
data = {'streamer_name': ['프로젝트아이', '스텔라이브', '이세계아이돌']} 
df = pd.DataFrame(data)

headers = {
    "User-Agent": "Mozilla/5.0 (Linux; Android 10; SM-G981B) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/80.0.3987.162 Mobile Safari/537.36"
}

# 2. 첫 번째 아이디만 냅다 물어오는 함수!
def get_first_cafe_id(streamer_name):
    search_keyword = urllib.parse.quote(f"{streamer_name} 공식 팬카페") 
    url = f"https://m.search.naver.com/search.naver?where=m_cafe&query={search_keyword}"
    
    try:
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # 검색 결과의 링크들을 순서대로 확인합니다.
        for a_tag in soup.find_all('a', href=True):
            href = a_tag['href']
            
            if 'cafe.naver.com/' in href:
                match = re.search(r'cafe\.naver\.com/([a-zA-Z0-9_]+)', href)
                if match:
                    cafe_id = match.group(1)
                    
                    # 네이버 잡동사니 링크만 아니면...
                    if len(cafe_id) > 2 and 'search' not in cafe_id:
                        # 💥 핵심: 찾자마자 고민 안 하고 바로 리턴해버림!!! 💥
                        return cafe_id 
                        
        return None
    except Exception as e:
        print(f"오류: {e}")
        return None

print("🏎️ 노빠꾸 직진 크롤러 출동합니다...\n")
found_ids = []

# 3. 데이터프레임 순회하며 바로바로 꽂아넣기
for name in df['streamer_name']:
    cafe_id = get_first_cafe_id(name)
    found_ids.append(cafe_id)
    print(f"🎯 [{name}] ➡️ 1빠따 아이디: {cafe_id if cafe_id else '실패 😭'}")
    time.sleep(1) # 그래도 네이버 눈치는 봐야 하니 1초 휴식

# 4. 결과 저장!
df['cafe_id'] = found_ids
print("\n🎉 수집 완료! 이대로 바로 저장합니다!")
print(df)

# df.to_csv('cime_fast_mapped.csv', index=False, encoding='utf-8-sig')

🏎️ 노빠꾸 직진 크롤러 출동합니다...

🎯 [프로젝트아이] ➡️ 1빠따 아이디: projectiofficial
🎯 [스텔라이브] ➡️ 1빠따 아이디: tteokbokk1
🎯 [이세계아이돌] ➡️ 1빠따 아이디: steamindiegame

🎉 수집 완료! 이대로 바로 저장합니다!
  streamer_name           cafe_id
0        프로젝트아이  projectiofficial
1         스텔라이브        tteokbokk1
2        이세계아이돌    steamindiegame


In [5]:
import os
import random
import time
import re
import urllib.parse
import pandas as pd
import requests

# 1. 파일 설정 (인코딩 에러 방지 포함)
file_path = 'final_streamer_dataset_1p_122p.csv'

try:
    df = pd.read_csv(file_path, encoding='utf-8-sig')
except UnicodeDecodeError:
    try:
        df = pd.read_csv(file_path, encoding='cp949')
    except:
        df = pd.read_csv(file_path, encoding='utf-8')

unique_streamers = df['streamer_name'].dropna().unique().tolist()
target_streamers = unique_streamers[500 : 1000] 

# 2. 변장용 가면 리스트
ua_list = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Mozilla/5.0 (iPhone; CPU iPhone OS 17_3_1 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.2 Mobile/15E148 Safari/604.1"
]

def get_cafe_id_monitor(streamer_name):
    search_keyword = urllib.parse.quote(f"{streamer_name} 공식 팬카페")
    url = f"https://m.search.naver.com/search.naver?where=m_cafe&query={search_keyword}"
    
    headers = {
        "User-Agent": random.choice(ua_list),
        "Referer": "https://m.naver.com/"
    }
    
    try:
        res = requests.get(url, headers=headers, timeout=10)
        # 🌟 상태 코드를 아이디와 함께 반환하도록 수정!
        status = res.status_code
        
        if status == 200:
            ids = re.findall(r'cafe\.naver\.com/([a-zA-Z0-9_]+)', res.text)
            candidates = [cid for cid in ids if len(cid) > 3 and cid not in ['naver', 'search', 'my', 'section']]
            cafe_id = candidates[0] if candidates else None
            return status, cafe_id
        else:
            return status, None # 403 등이 뜨면 바로 상태 코드 반환
    except:
        return "ERROR", None

# 3. 메인 루프 (실시간 관제탑 가동!)
results = []
print(f"📡 [관제 시작] 산사 선샌니, 실시간으로 상태를 보고합니다! (Target: 500명)")
print("-" * 50)

for i, name in enumerate(target_streamers):
    status, cafe_id = get_cafe_id_monitor(name)
    
    # 화면 출력 (상태 코드를 맨 앞에!)
    # ✅ 200이면 안심, ❌ 403이면 즉시 비상사태!
    status_icon = "✅" if status == 200 else "❌"
    print(f"{status_icon} [{status}] {i+1}/500 | {name} ➡️ {cafe_id if cafe_id else '찾지 못함'}")
    
    if status == 403:
        print("\n🚨 [경보] 403 차단 발생! 즉시 중단합니다. 핫스팟 IP를 새로 고쳐주세요!")
        break
        
    results.append({'streamer_name': name, 'cafe_id': cafe_id})
    
    # 넉넉한 휴식 시간 (핫스팟 보호)
    time.sleep(random.uniform(4.0, 8.0))

# 저장
pd.DataFrame(results).to_csv('mapping_test_500_1000_monitor.csv', index=False, encoding='utf-8-sig')
print("\n🎊 수집 종료!")

📡 [관제 시작] 산사 선샌니, 실시간으로 상태를 보고합니다! (Target: 500명)
--------------------------------------------------
✅ [200] 1/500 | 단피아 ➡️ 찾지 못함
✅ [200] 2/500 | 단행 DanHaeng ➡️ 찾지 못함
✅ [200] 3/500 | 달 콤 ➡️ cursday
✅ [200] 4/500 | 달다람 Dal ➡️ 찾지 못함
✅ [200] 5/500 | 달달하랑I ➡️ 찾지 못함
✅ [200] 6/500 | 달라찌 ➡️ 찾지 못함
✅ [200] 7/500 | 달문양 ➡️ moomoo
✅ [200] 8/500 | 달미호 ➡️ mafca
✅ [200] 9/500 | 달빛아래 키레얀 ➡️ 찾지 못함
✅ [200] 10/500 | 달빛여울 ➡️ 찾지 못함
✅ [200] 11/500 | 달이레오 ➡️ 찾지 못함
✅ [200] 12/500 | 달이안 ➡️ 찾지 못함
✅ [200] 13/500 | 달자기 ➡️ 찾지 못함
✅ [200] 14/500 | 담 일 ➡️ tvarotti1002
✅ [200] 15/500 | 담아서 ➡️ dama0804
✅ [200] 16/500 | 당당구리1220 ➡️ 찾지 못함
✅ [200] 17/500 | 당돌 섹시 상진 ➡️ 찾지 못함
✅ [200] 18/500 | 당컴남 ➡️ 찾지 못함
✅ [200] 19/500 | 대기쟝 ➡️ 찾지 못함
✅ [200] 20/500 | 대댐 ➡️ 찾지 못함
✅ [200] 21/500 | 대드냥 ➡️ moomoo
✅ [200] 22/500 | 대장7 ➡️ mrtrot2official
✅ [200] 23/500 | 댄버Daenbeo ➡️ 찾지 못함
✅ [200] 24/500 | 댕댕댕댕댕댕댕댕 ➡️ photatosujebi
✅ [200] 25/500 | 댕댕한댕식씨 ➡️ 찾지 못함
✅ [200] 26/500 | 댕아요 ➡️ daengayo
✅ [200] 27/500 | 더 밍 ➡️ eggkim
✅ [200] 28/500 | 더

KeyboardInterrupt: 

In [3]:
import os
import random
import time
import re
import urllib.parse
import pandas as pd
import requests

# ==========================================
# 1. 데이터 불러오기 (1번부터 다시 드가자!)
# ==========================================
file_path = 'softcone_streamer_dataset1.csv'

try:
    df = pd.read_csv(file_path, encoding='utf-8-sig')
except UnicodeDecodeError:
    try:
        df = pd.read_csv(file_path, encoding='cp949')
    except:
        df = pd.read_csv(file_path, encoding='utf-8')

# 전체 리스트 (이번엔 슬라이싱 없이 통째로!)
unique_streamers = df['streamer_name'].dropna().unique().tolist()
total_count = len(unique_streamers)

print(f"📊 [리셋 완료] 1번부터 {total_count}번까지 처음부터 자동 수집을 시작합니다!")

# ==========================================
# 2. 일꾼 함수 및 변장 도구
# ==========================================
ua_list = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
    "Mozilla/5.0 (iPhone; CPU iPhone OS 17_3_1 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.2 Mobile/15E148 Safari/604.1"
]

def get_cafe_id_monitor(streamer_name):
    search_keyword = urllib.parse.quote(f"{streamer_name} 공식 팬카페")
    url = f"https://m.search.naver.com/search.naver?where=m_cafe&query={search_keyword}"
    headers = {
        "User-Agent": random.choice(ua_list), 
        "Referer": "https://m.naver.com/"
    }
    try:
        res = requests.get(url, headers=headers, timeout=10)
        status = res.status_code
        if status == 200:
            ids = re.findall(r'cafe\.naver\.com/([a-zA-Z0-9_]+)', res.text)
            candidates = [cid for cid in ids if len(cid) > 3 and cid not in ['naver', 'search', 'my', 'section']]
            return status, (candidates[0] if candidates else None)
        else:
            return status, None
    except:
        return "ERROR", None

# ==========================================
# 3. 500개씩 자동 루프 가동 (Starting from Index 0)
# ==========================================
chunk_size = 500
stop_all = False

for i in range(493, total_count, chunk_size):
    if stop_all: break
    
    # 이번 묶음 슬라이싱
    chunk = unique_streamers[i : i + chunk_size]
    
    # 파일 이름 및 번호 계산 (사람 기준)
    current_start = i + 1
    current_end = min(i + chunk_size, total_count)
    file_name = f'mapping_{current_start}_{current_end}_monitor.csv'
    
    # 🌟 체크포인트: 이미 완료된 파일이면 건너뛰기
    if os.path.exists(file_name):
        print(f"⏩ {current_start}~{current_end} 묶음은 이미 완료되어 건너뜁니다.")
        continue

    print(f"\n🚀 {current_start}번부터 {current_end}번까지 묶음 수집 시작!")
    results = []
    
    for j, name in enumerate(chunk):
        status, cafe_id = get_cafe_id_monitor(name)
        
        # 선샌니가 좋아하시는 그 출력 멘트!
        status_icon = "✅" if status == 200 else "❌"
        real_num = i + j + 1
        print(f"{status_icon} [{status}] {real_num}/{total_count} | {name} ➡️ {cafe_id if cafe_id else '찾지 못함'}")
        
        # 403 차단 시 즉시 중단
        if status == 403:
            print(f"\n🚨 [비상] {real_num}번에서 403 차단 발생! 중단 후 IP를 변경하세요.")
            stop_all = True
            break
            
        results.append({'streamer_name': name, 'cafe_id': cafe_id})
        
        # 안전한 지연 시간 (4~8초)
        time.sleep(random.uniform(4.0, 8.0))
    
    # 한 묶음 끝날 때마다 중간 저장
    if results:
        pd.DataFrame(results).to_csv(file_name, index=False, encoding='utf-8-sig')
        print(f"💾 {file_name} 저장 완료!")
        
        # 묶음 사이에는 좀 더 길게 쉬어주기
        if not stop_all:
            print("💤 다음 묶음을 준비하며 30초간 휴식합니다...")
            time.sleep(30)

print("\n🎊 모든 자동화 수집 작업이 종료되었습니다! 선샌니 고생하셨어요!")

📊 [리셋 완료] 1번부터 4943번까지 처음부터 자동 수집을 시작합니다!

🚀 494번부터 993번까지 묶음 수집 시작!
✅ [200] 494/4943 | 다해달 ➡️ misskim0308
✅ [200] 495/4943 | 다희닝 ➡️ 찾지 못함
✅ [200] 496/4943 | 단 님 ➡️ daesunggong21
✅ [200] 497/4943 | 단마브 ➡️ raktoeic
✅ [200] 498/4943 | 단미미호 ➡️ mafca
✅ [200] 499/4943 | 단이진 ➡️ onmyoji2017
✅ [200] 500/4943 | 단코퐁 ➡️ 찾지 못함
✅ [200] 501/4943 | 단피아 ➡️ 찾지 못함
✅ [200] 502/4943 | 단행 DanHaeng ➡️ 찾지 못함
✅ [200] 503/4943 | 달 콤 ➡️ cursday
✅ [200] 504/4943 | 달다람 Dal ➡️ 찾지 못함
✅ [200] 505/4943 | 달달하랑I ➡️ 찾지 못함
✅ [200] 506/4943 | 달라찌 ➡️ 찾지 못함
✅ [200] 507/4943 | 달문양 ➡️ honkaistarrail
✅ [200] 508/4943 | 달미호 ➡️ mafca
✅ [200] 509/4943 | 달빛아래 키레얀 ➡️ 찾지 못함
✅ [200] 510/4943 | 달빛여울 ➡️ 찾지 못함
✅ [200] 511/4943 | 달이레오 ➡️ 찾지 못함
✅ [200] 512/4943 | 달이안 ➡️ 찾지 못함
✅ [200] 513/4943 | 달자기 ➡️ 찾지 못함
✅ [200] 514/4943 | 담 일 ➡️ tvarotti1002
✅ [200] 515/4943 | 담아서 ➡️ dama0804
✅ [200] 516/4943 | 당당구리1220 ➡️ 찾지 못함
✅ [200] 517/4943 | 당돌 섹시 상진 ➡️ 찾지 못함
✅ [200] 518/4943 | 당컴남 ➡️ 찾지 못함
✅ [200] 519/4943 | 대기쟝 ➡️ 찾지 못함
✅ [200] 520/4943 | 대댐 ➡

KeyboardInterrupt: 

import pandas as pd
import requests
from bs4 import BeautifulSoup
import urllib.parse
import re
import time

# 1. 선샌니의 DB(CSV 파일) 불러오기!
file_path = '샘플_소프트콘.csv'

# 혹시 한글 인코딩 에러가 날까 봐 보험용으로 try-except를 걸어둡니다.
try:
    df = pd.read_csv(file_path, encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(file_path, encoding='cp949')

# 🌟 핵심 스킬 1: 중복된 스트리머 이름 제거하고 고유한 이름만 리스트로 뽑기!
unique_streamers = df['streamer_name'].dropna().unique()
print(f"📊 총 {len(df)}줄의 데이터에서, {len(unique_streamers)}명의 고유 스트리머를 찾아냈습니다!\n")

headers = {
    "User-Agent": "Mozilla/5.0 (Linux; Android 10; SM-G981B) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/80.0.3987.162 Mobile Safari/537.36"
}

# (아까 만든 노빠꾸 1빠따 크롤러 함수!)
def get_first_cafe_id(streamer_name):
    search_keyword = urllib.parse.quote(f"{streamer_name} 공식 팬카페") 
    url = f"https://m.search.naver.com/search.naver?where=m_cafe&query={search_keyword}"
    
    try:
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        for a_tag in soup.find_all('a', href=True):
            href = a_tag['href']
            if 'cafe.naver.com/' in href:
                match = re.search(r'cafe\.naver\.com/([a-zA-Z0-9_]+)', href)
                if match:
                    cafe_id = match.group(1)
                    if len(cafe_id) > 2 and 'search' not in cafe_id:
                        return cafe_id 
        return None
    except Exception as e:
        print(f"오류: {e}")
        return None

# 3. 고유 스트리머 이름으로만 검색 시작!
print("🏎️ 노빠꾸 직진 크롤러 출동합니다...\n")
mapping_data = [] # 이름 - 카페아이디 짝꿍을 저장할 바구니

for name in unique_streamers:
    cafe_id = get_first_cafe_id(name)
    mapping_data.append({'streamer_name': name, 'cafe_id': cafe_id})
    print(f"🎯 [{name}] ➡️ 1빠따 아이디: {cafe_id if cafe_id else '실패 😭'}")
    time.sleep(1) # 차단 방지 매너 1초!

# 4. 찾아온 짝꿍 데이터를 데이터프레임으로 변환
mapping_df = pd.DataFrame(mapping_data)

# 🌟 핵심 스킬 2: 원본 DB와 매핑 테이블 합치기 (SQL의 LEFT JOIN과 똑같아요!)
final_df = pd.merge(df, mapping_df, on='streamer_name', how='left')

# 5. 최종 결과 저장!
final_df.to_csv('최종_소프트콘_카페추가.csv', index=False, encoding='utf-8-sig')

print("\n🎉 수집 및 병합 완료!")
print("폴더에 생성된 '최종_소프트콘_카페추가.csv'를 확인해 보세요! 맨 오른쪽에 cafe_id 열이 쫙 붙어있을 겁니다!")
# --- (이전 코드 맨 아래에 이어서 붙여넣기) ---

# 🌟 보너스 스킬: 성공/실패 인원수 집계하기
# mapping_df를 기준으로 고유 스트리머 중 몇 명을 성공/실패했는지 셉니다.
success_count = mapping_df['cafe_id'].notna().sum()
fail_count = mapping_df['cafe_id'].isna().sum()

print("="*40)
print("📊 [크롤링 최종 결과 보고서]")
print(f"✅ 성공 (카페 찾음): {success_count}명")
print(f"❌ 실패 (카페 못찾음): {fail_count}명")
print("="*40)

# 만약 실패한 스트리머 이름만 쏙 뽑아서 보고 싶다면?
failed_streamers = mapping_df[mapping_df['cafe_id'].isna()]['streamer_name'].tolist()
if fail_count > 0:
    print(f"😭 못 찾은 스트리머 명단: {failed_streamers}")

import pandas as pd
import requests
from bs4 import BeautifulSoup
import urllib.parse
import re
import time

# --- 기존 코드와 동일 ---
file_path = 'final_streamer_dataset_1p_122p.csv'
try:
    df = pd.read_csv(file_path, encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(file_path, encoding='cp949')

unique_streamers = df['streamer_name'].dropna().unique()
print(f"📊 총 {len(unique_streamers)}명의 고유 스트리머 수집 시작!\n")

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# 🌟 업그레이드된 스마트 크롤러: 매니저 신원조회 기능 탑재!
def get_cafe_id_smart(streamer_name):
    search_keyword = urllib.parse.quote(f"{streamer_name} 공식 팬카페") 
    url = f"https://m.search.naver.com/search.naver?where=m_cafe&query={search_keyword}"
    
    try:
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # 1. 상위 3개 후보 카페 아이디 수집
        candidates = []
        for a_tag in soup.find_all('a', href=True):
            href = a_tag['href']
            if 'cafe.naver.com/' in href:
                match = re.search(r'cafe\.naver\.com/([a-zA-Z0-9_]+)', href)
                if match:
                    cafe_id = match.group(1)
                    if len(cafe_id) > 2 and 'search' not in cafe_id and cafe_id not in candidates:
                        candidates.append(cafe_id)
                    if len(candidates) == 3: # 딱 3개까지만 용의선상에 올립니다.
                        break
                        
        if not candidates:
            return None

        # 2. [플랜 B] 후보 카페들에 직접 들어가서 매니저 이름 확인!
        # 검색어(스트리머명)의 공백을 없애고 소문자로 만들어서 비교 준비
        clean_streamer_name = streamer_name.replace(" ", "").lower()
        
        for cafe_id in candidates:
            cafe_url = f"https://cafe.naver.com/{cafe_id}" # 매니저 확인은 PC주소가 편해요!
            cafe_res = requests.get(cafe_url, headers=headers)
            cafe_soup = BeautifulSoup(cafe_res.text, 'html.parser')
            
            manager_name = ""
            # 이전에 썼던 '불도저' 로직 재등장! 카페 정보란에서 '매니저' 텍스트 찾기
            for li in cafe_soup.select('#cafe-info-data > ul > li'):
                text_content = li.text.strip().replace('\n', '').replace(' ', '')
                if '매니저' in text_content:
                    manager_name = text_content.replace('매니저', '').lower()
                    break
            
            # 매니저 이름에 스트리머 이름이 포함되어 있다면?!
            if clean_streamer_name in manager_name:
                print(f"    ✨ [검거 완료] 매니저({manager_name}) 일치! -> {cafe_id}")
                return cafe_id # 정답 찾았으니 즉시 리턴!

        # 3. 만약 3개 다 매니저 이름이 안 맞으면? 
        # (스트리머가 부계정을 쓰거나 찐팬이 매니저인 경우)
        # -> 기존 80%의 성공률을 지키기 위해 그냥 제일 첫 번째(1빠따) 카페를 던져줍니다.
        return candidates[0]
        
    except Exception as e:
        print(f"오류: {e}")
        return None

# --- 실행 파트 ---
print("🕵️‍♂️ 스마트 탐정단 출동합니다...\n")
mapping_data = []

for name in unique_streamers:
    cafe_id = get_cafe_id_smart(name)
    mapping_data.append({'streamer_name': name, 'cafe_id': cafe_id})
    print(f"🎯 [{name}] ➡️ 최종 선택 아이디: {cafe_id if cafe_id else '실패 😭'}")
    time.sleep(1) # 차단 방지 매너 1초!

# 합치고 저장하는 로직
mapping_df = pd.DataFrame(mapping_data)
final_df = pd.merge(df, mapping_df, on='streamer_name', how='left')
final_df.to_csv('최종_소프트콘_매니저검증판.csv', index=False, encoding='utf-8-sig')

# 결과 보고서
success_count = mapping_df['cafe_id'].notna().sum()
fail_count = mapping_df['cafe_id'].isna().sum()

print("\n" + "="*40)
print("📊 [크롤링 최종 결과 보고서]")
print(f"✅ 성공 (카페 찾음): {success_count}명")
print(f"❌ 실패 (카페 못찾음): {fail_count}명")
print("="*40)

# 🔍 '쇼 카'님 카페 찾기 성공 여부 확인 코드

target_name = '강지' 

# 1. 매핑 결과(mapping_df)에서 해당 스트리머 찾기
# (참고: mapping_df는 고유 스트리머 이름과 카페ID가 담긴 표입니다)
result = mapping_df[mapping_df['streamer_name'] == target_name]

if not result.empty:
    cafe_id = result.iloc[0]['cafe_id']
    
    if pd.notna(cafe_id):
        print(f"✨ [검거 성공!] '{target_name}'님의 카페 아이디를 찾았습니다!")
        print(f"📌 카페 아이디: {cafe_id}")
        print(f"🔗 바로가기: https://cafe.naver.com/{cafe_id}")
    else:
        print(f"😭 [검거 실패...] '{target_name}'님은 아쉽게도 실패한 11,441명 중 한 분이네요.")
        print(f"💡 이유 추정: 네이버 차단으로 검색 결과가 안 떴거나, 카페를 운영하지 않으실 수 있어요.")
else:
    print(f"❓ '{target_name}'님은 애초에 수집 대상 명단에 없었던 것 같아요. 이름을 다시 확인해 보세요!")

In [2]:
import os
import random
import time
import re
import urllib.parse
import pandas as pd
import requests
from bs4 import BeautifulSoup

# ==========================================
# 1. 데이터 불러오기 (우리 데이터 로드!)
# ==========================================
file_path = 'softcone_streamer_dataset1.csv.csv' # 선샌니의 찐 데이터 파일명

try:
    df = pd.read_csv(file_path, encoding='utf-8-sig')
except:
    df = pd.read_csv(file_path, encoding='cp949')

# 고유 스트리머 이름만 추출 (중복 제거)
unique_streamers = df['streamer_name'].dropna().unique().tolist()
print(f"📊 총 {len(unique_streamers)}명의 스트리머 수집을 시작합니다.")

# ==========================================
# 2. 필요한 함수들 정의 (일꾼들 준비!)
# ==========================================
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# [쪼개기 함수]
def chunk_list(lst, n):
    for i in range(1494, len(lst), n):
        yield lst[i:i + n]

# [스마트 크롤러 함수]
def get_cafe_id_smart(streamer_name):
    search_keyword = urllib.parse.quote(f"{streamer_name} 공식 팬카페") 
    url = f"https://m.search.naver.com/search.naver?where=m_cafe&query={search_keyword}"
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')
        candidates = []
        for a_tag in soup.find_all('a', href=True):
            href = a_tag['href']
            if 'cafe.naver.com/' in href:
                match = re.search(r'cafe\.naver\.com/([a-zA-Z0-9_]+)', href)
                if match and match.group(1) not in candidates:
                    candidates.append(match.group(1))
            if len(candidates) == 3: break
        
        if not candidates: return None

        # 매니저 신원조회 (플랜 B)
        clean_name = streamer_name.replace(" ", "").lower()
        for cafe_id in candidates:
            try:
                cafe_res = requests.get(f"https://cafe.naver.com/{cafe_id}", headers=headers, timeout=5)
                if clean_name in cafe_res.text.lower(): # 불도저식 텍스트 검색
                    return cafe_id
            except: continue
        return candidates[0]
    except:
        return None

# ==========================================
# 3. 쪼개서 자동 실행 (컨베이어 벨트 가동!)
# ==========================================
chunk_size = 500
chunks = list(chunk_list(unique_streamers, chunk_size))

for idx, chunk in enumerate(chunks):
    file_name = f'mapping_part_{idx+1}.csv'
    
    # 이미 완료된 파일이면 건너뛰기
    if os.path.exists(file_name):
        print(f"⏩ {idx+1}번 묶음({file_name})은 이미 완료되어 건너뜁니다.")
        continue
        
    print(f"🚀 {idx+1}/{len(chunks)}번째 묶음 수집 시작...")
    results = []
    
    for name in chunk:
        cafe_id = get_cafe_id_smart(name)
        results.append({'streamer_name': name, 'cafe_id': cafe_id})
        print(f"🎯 [{name}] -> {cafe_id}")
        time.sleep(random.uniform(1.5, 3.0)) # 랜덤 지연 시간
        
    # 묶음 단위 중간 저장
    pd.DataFrame(results).to_csv(file_name, index=False, encoding='utf-8-sig')
    print(f"✅ {idx+1}번 완료! 30초 쉬고 다음으로 넘어갑니다...")
    time.sleep(30)

print("\n🎊 모든 작업이 끝났습니다, 산사 선샌니! 이제 폴더에 생긴 여러 개의 csv를 합치기만 하면 돼요!")

FileNotFoundError: [Errno 2] No such file or directory: 'softcone_streamer_dataset1.csv.csv'

In [2]:
import os
import random
import time
import re
import urllib.parse
import pandas as pd
import requests

# ==========================================
# 1. 데이터 불러오기 (1번부터 다시 드가자!)
# ==========================================
file_path = 'softcone_streamer_dataset1.csv'

try:
    df = pd.read_csv(file_path, encoding='utf-8-sig')
except UnicodeDecodeError:
    try:
        df = pd.read_csv(file_path, encoding='cp949')
    except:
        df = pd.read_csv(file_path, encoding='utf-8')

# 전체 리스트 (이번엔 슬라이싱 없이 통째로!)
unique_streamers = df['streamer_name'].dropna().unique().tolist()
total_count = len(unique_streamers)

print(f"📊 [리셋 완료] 1번부터 {total_count}번까지 처음부터 자동 수집을 시작합니다!")

# ==========================================
# 2. 일꾼 함수 및 변장 도구
# ==========================================
ua_list = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
    "Mozilla/5.0 (iPhone; CPU iPhone OS 17_3_1 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.2 Mobile/15E148 Safari/604.1"
]

def get_cafe_id_monitor(streamer_name):
    search_keyword = urllib.parse.quote(f"{streamer_name} 공식 팬카페")
    url = f"https://m.search.naver.com/search.naver?where=m_cafe&query={search_keyword}"
    headers = {
        "User-Agent": random.choice(ua_list), 
        "Referer": "https://m.naver.com/"
    }
    try:
        res = requests.get(url, headers=headers, timeout=10)
        status = res.status_code
        if status == 200:
            ids = re.findall(r'cafe\.naver\.com/([a-zA-Z0-9_]+)', res.text)
            candidates = [cid for cid in ids if len(cid) > 3 and cid not in ['naver', 'search', 'my', 'section']]
            return status, (candidates[0] if candidates else None)
        else:
            return status, None
    except:
        return "ERROR", None

# ==========================================
# 3. 500개씩 자동 루프 가동 (Starting from Index 0)
# ==========================================
chunk_size = 500
stop_all = False

for i in range(2498, total_count, chunk_size):
    if stop_all: break
    
    # 이번 묶음 슬라이싱
    chunk = unique_streamers[i : i + chunk_size]
    
    # 파일 이름 및 번호 계산 (사람 기준)
    current_start = i + 1
    current_end = min(i + chunk_size, total_count)
    file_name = f'mapping_{current_start}_{current_end}_monitor.csv'
    
    # 🌟 체크포인트: 이미 완료된 파일이면 건너뛰기
    if os.path.exists(file_name):
        print(f"⏩ {current_start}~{current_end} 묶음은 이미 완료되어 건너뜁니다.")
        continue

    print(f"\n🚀 {current_start}번부터 {current_end}번까지 묶음 수집 시작!")
    results = []
    
    for j, name in enumerate(chunk):
        status, cafe_id = get_cafe_id_monitor(name)
        
        # 선샌니가 좋아하시는 그 출력 멘트!
        status_icon = "✅" if status == 200 else "❌"
        real_num = i + j + 1
        print(f"{status_icon} [{status}] {real_num}/{total_count} | {name} ➡️ {cafe_id if cafe_id else '찾지 못함'}")
        
        # 403 차단 시 즉시 중단
        if status == 403:
            print(f"\n🚨 [비상] {real_num}번에서 403 차단 발생! 중단 후 IP를 변경하세요.")
            stop_all = True
            break
            
        results.append({'streamer_name': name, 'cafe_id': cafe_id})
        
        # 안전한 지연 시간 (4~8초)
        time.sleep(random.uniform(4.0, 8.0))
    
    # 한 묶음 끝날 때마다 중간 저장
    if results:
        pd.DataFrame(results).to_csv(file_name, index=False, encoding='utf-8-sig')
        print(f"💾 {file_name} 저장 완료!")
        
        # 묶음 사이에는 좀 더 길게 쉬어주기
        if not stop_all:
            print("💤 다음 묶음을 준비하며 30초간 휴식합니다...")
            time.sleep(30)

print("\n🎊 모든 자동화 수집 작업이 종료되었습니다! 선샌니 고생하셨어요!")

📊 [리셋 완료] 1번부터 4943번까지 처음부터 자동 수집을 시작합니다!

🚀 2499번부터 2998번까지 묶음 수집 시작!
✅ [200] 2499/4943 | 마앙 ➡️ 찾지 못함
✅ [200] 2500/4943 | 메이쥰 ➡️ 찾지 못함
✅ [200] 2501/4943 | 멜제 ➡️ 찾지 못함
✅ [200] 2502/4943 | 멸화랑 ➡️ 찾지 못함
✅ [200] 2503/4943 | 모서리 ➡️ khantata
✅ [200] 2504/4943 | 무감 ➡️ 찾지 못함
✅ [200] 2505/4943 | 뮤차린 ➡️ 찾지 못함
✅ [200] 2506/4943 | 미나미 레이 ➡️ 찾지 못함
✅ [200] 2507/4943 | 미니 ➡️ beautifer
✅ [200] 2508/4943 | 미요 ➡️ miyocute
✅ [200] 2509/4943 | 민하 ➡️ 찾지 못함
✅ [200] 2510/4943 | 박찬물 ➡️ 찾지 못함
✅ [200] 2511/4943 | 반토 ➡️ banto425
✅ [200] 2512/4943 | 밥와지 ➡️ 찾지 못함
✅ [200] 2513/4943 | 백효 ➡️ 찾지 못함
✅ [200] 2514/4943 | 블랙0맘바 ➡️ 찾지 못함
✅ [200] 2515/4943 | 비도 ➡️ bver99
✅ [200] 2516/4943 | 비탄 ➡️ elsea
✅ [200] 2517/4943 | 삐뇽 ➡️ bbinyong
✅ [200] 2518/4943 | 서라빈 ➡️ moomoo
✅ [200] 2519/4943 | 선 율 ➡️ 2chanwon
✅ [200] 2520/4943 | 설산범 ➡️ 찾지 못함
✅ [200] 2521/4943 | 세 미로 ➡️ 찾지 못함
✅ [200] 2522/4943 | 세리스 ➡️ baedon
✅ [200] 2523/4943 | 세분자 ➡️ 찾지 못함
✅ [200] 2524/4943 | 소람 ➡️ 찾지 못함
✅ [200] 2525/4943 | 소사량 ➡️ sosalyang55267010101
✅ [200]

In [3]:
import pandas as pd
import glob
import os

# ==========================================
# 1. 조각 파일들 싹쓸이 모으기!
# ==========================================
# 'mapping_'으로 시작하고 '_monitor.csv'로 끝나는 모든 파일을 찾습니다.
# 만약 파일 이름을 다르게 저장하셨다면 이 패턴을 수정해 주세요!
file_pattern = 'mapping_*_monitor.csv' 
file_list = glob.glob(file_pattern)

print(f"📦 총 {len(file_list)}개의 조각 파일을 발견했습니다!\n")

if not file_list:
    print("🚨 앗! 합칠 파일이 하나도 안 보여요. 파일이 있는 폴더에서 실행한 게 맞는지 확인해 주세요!")
else:
    # ==========================================
    # 2. 파일 하나씩 열어서 리스트에 담기
    # ==========================================
    df_list = []
    for file in file_list:
        try:
            # 한글 인코딩 에러 철벽 방어!
            df = pd.read_csv(file, encoding='utf-8-sig')
        except UnicodeDecodeError:
            try:
                df = pd.read_csv(file, encoding='cp949')
            except:
                df = pd.read_csv(file, encoding='utf-8')
        
        df_list.append(df)
        print(f"✔️ [{file}] 무사히 읽어왔습니다! (데이터 {len(df)}개)")

    # ==========================================
    # 3. 하나로 찰흙처럼 뭉치고 중복 제거하기
    # ==========================================
    # concat으로 위아래로 길게 이어 붙입니다.
    merged_df = pd.concat(df_list, ignore_index=True)
    
    # 껐다 켰다 하면서 혹시 겹친 데이터가 있을 수 있으니, 스트리머 이름 기준으로 중복을 제거합니다.
    before_count = len(merged_df)
    merged_df = merged_df.drop_duplicates(subset=['streamer_name'], keep='last')
    after_count = len(merged_df)
    
    if before_count != after_count:
        print(f"\n🧹 중복된 데이터 {before_count - after_count}개를 깔끔하게 청소했습니다!")

    # ==========================================
    # 4. 최종 마스터 파일 저장!
    # ==========================================
    final_filename = 'final_mapping_master.csv'
    merged_df.to_csv(final_filename, index=False, encoding='utf-8-sig')
    
    print("\n" + "="*50)
    print("🎊 [병합 대성공] 모든 조각이 하나로 합쳐졌습니다! 🎊")
    print(f"📊 최종 수집된 카페 정보: 총 {len(merged_df)}명")
    print(f"💾 마스터 파일명: {final_filename}")
    print("="*50)

📦 총 10개의 조각 파일을 발견했습니다!

✔️ [mapping_1495_1994_monitor.csv] 무사히 읽어왔습니다! (데이터 500개)
✔️ [mapping_1997_2496_monitor.csv] 무사히 읽어왔습니다! (데이터 500개)
✔️ [mapping_1_500_monitor.csv] 무사히 읽어왔습니다! (데이터 492개)
✔️ [mapping_2499_2998_monitor.csv] 무사히 읽어왔습니다! (데이터 500개)
✔️ [mapping_2999_3498_monitor.csv] 무사히 읽어왔습니다! (데이터 500개)
✔️ [mapping_3499_3998_monitor.csv] 무사히 읽어왔습니다! (데이터 500개)
✔️ [mapping_3999_4498_monitor.csv] 무사히 읽어왔습니다! (데이터 500개)
✔️ [mapping_4499_4943_monitor.csv] 무사히 읽어왔습니다! (데이터 445개)
✔️ [mapping_494_993_monitor.csv] 무사히 읽어왔습니다! (데이터 500개)
✔️ [mapping_994_1493_monitor.csv] 무사히 읽어왔습니다! (데이터 500개)

🎊 [병합 대성공] 모든 조각이 하나로 합쳐졌습니다! 🎊
📊 최종 수집된 카페 정보: 총 4937명
💾 마스터 파일명: final_mapping_master.csv


In [5]:
import requests
from bs4 import BeautifulSoup
import re

# 1. 테스트할 카페 아이디 (다른 카페를 테스트하고 싶으면 여기만 바꾸세요!)
test_cafe_id = "tteokbokk1" 
url = f"https://cafe.naver.com/{test_cafe_id}" 

# 2. 브라우저 위장
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Referer": "https://www.naver.com/"
}

print(f"🕵️‍♂️ '{test_cafe_id}' 카페(PC버전) 불도저 엔진 테스트 시작!\n" + "-"*40)

try:
    res = requests.get(url, headers=headers, timeout=10)
    print(f"📡 응답 상태: {res.status_code}")
    
    if res.status_code == 200:
        soup = BeautifulSoup(res.text, 'html.parser')
        
        # ==========================================
        # [1] 멤버 수 (안정적인 기존 로직)
        # ==========================================
        member_element = soup.select_one('.mem-cnt-info em') 
        if member_element:
            member_count = int(re.sub(r'[^0-9]', '', member_element.text))
            print(f"✅ 카페 가입자 수 획득 성공!: {member_count:,}명")
        else:
            print("❌ 멤버 수를 찾지 못했습니다.")
            
        # ==========================================
        # [2] 전체글 수 (선샌니의 불도저 로직 🚜)
        # ==========================================
        total_posts = 0
        all_li_tags = soup.find_all('li') 
        
        for li in all_li_tags:
            text_content = li.text.strip().replace('\n', '').replace(' ', '')
            if '전체글' in text_content:
                numbers = re.findall(r'[\d,]+', text_content)
                if numbers:
                    total_posts = int(numbers[0].replace(',', ''))
                    break # 찾았으니 미련 없이 탈출!
                    
        if total_posts > 0:
            print(f"✅ 전체 게시글 수 획득 성공!: {total_posts:,}개")
        else:
            print("❌ 전체 게시글 수를 찾지 못했습니다.")
            
    else:
        print(f"❌ 네이버가 거부했습니다. 상태 코드: {res.status_code}")

except Exception as e:
    print(f"❌ 접속 자체에 실패했습니다: {e}")

🕵️‍♂️ 'tteokbokk1' 카페(PC버전) 불도저 엔진 테스트 시작!
----------------------------------------
📡 응답 상태: 200
✅ 카페 가입자 수 획득 성공!: 253,160명
✅ 전체 게시글 수 획득 성공!: 844,451개


In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import random
import re
import os

# 1. 파일 불러오기
master_file = 'final_mapping_master.csv'
df = pd.read_csv(master_file, encoding='utf-8-sig')

target_df = df.dropna(subset=['cafe_id']).copy()
streamer_list = target_df['streamer_name'].tolist()
cafe_id_list = target_df['cafe_id'].tolist()
total_cafes = len(cafe_id_list)

print(f"📊 선샌니의 '불도저 로직' 탑재 완료! 총 {total_cafes}개 카페 털어봅시다!")

# 2. 일꾼 함수 (선샌니의 불도저 로직 완벽 이식!)
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Referer": "https://www.naver.com/"
}

def get_cafe_stats_pc(cafe_id):
    url = f"https://cafe.naver.com/{cafe_id}" 
    
    try:
        res = requests.get(url, headers=headers, timeout=10)
        status = res.status_code
        
        if status == 200:
            soup = BeautifulSoup(res.text, 'html.parser')
            member_count, post_count = "0", "0"
            
            # [1] 가입자 수 (성공했던 완벽한 코드)
            member_element = soup.select_one('.mem-cnt-info em') 
            if member_element:
                member_count = re.sub(r'[^0-9]', '', member_element.text)
                
            # [2] 전체 게시글 수 (선샌니의 진짜 찐 불도저 방식 🚜)
            all_li_tags = soup.find_all('li') 
            for li in all_li_tags:
                text_content = li.text.strip().replace('\n', '').replace(' ', '')
                if '전체글' in text_content:
                    numbers = re.findall(r'[\d,]+', text_content)
                    if numbers:
                        post_count = numbers[0].replace(',', '')
                        break # 찾았으니 미련 없이 탈출!
                        
            return status, member_count, post_count
        else:
            return status, None, None
    except:
        return "ERROR", None, None

# 3. 자동화 루프 가동 (관제탑 + 403 방어)
results = []
stop_all = False
chunk_size = 500

print("-" * 50)
print(f"🕵️‍♂️ [실시간 관제탑] 불도저 가동을 시작합니다!")

for i in range(0, total_cafes, chunk_size):
    if stop_all: break
    
    chunk_streamers = streamer_list[i : i + chunk_size]
    chunk_cafe_ids = cafe_id_list[i : i + chunk_size]
    
    current_start = i + 1
    current_end = min(i + chunk_size, total_cafes)
    # 파일 이름이 안 겹치게 final이라는 글자를 하나 넣었어요!
    file_name = f'cafe_stats_final_{current_start}_{current_end}.csv'
    
    if os.path.exists(file_name):
        print(f"⏩ {current_start}~{current_end} 묶음은 이미 완료되어 건너뜁니다.")
        continue

    print(f"\n🚀 {current_start}번부터 침투 시작!")
    chunk_results = []
    
    for j in range(len(chunk_cafe_ids)):
        s_name = chunk_streamers[j]
        c_id = chunk_cafe_ids[j]
        
        status, members, posts = get_cafe_stats_pc(c_id)
        
        # 선샌니가 눈여겨보시는 출력 화면!
        status_icon = "✅" if status == 200 else "❌"
        real_num = i + j + 1
        print(f"{status_icon} [{status}] {real_num}/{total_cafes} | {s_name}({c_id}) ➡️ 멤버: {members}명 / 글: {posts}개")
        
        if status == 403:
            print(f"\n🚨 [경보] 403 차단! 즉시 중단합니다. 핫스팟을 바꿔주세요.")
            stop_all = True
            break
            
        chunk_results.append({
            'streamer_name': s_name, 
            'cafe_id': c_id,
            'member_count': members,
            'post_count': posts
        })
        
        time.sleep(random.uniform(3.0, 6.0))
        
    if chunk_results:
        pd.DataFrame(chunk_results).to_csv(file_name, index=False, encoding='utf-8-sig')
        if not stop_all:
            print("💤 불도저 열 식히는 중... 15초 대기...")
            time.sleep(15)

print("\n🎊 진짜 리얼 찐 팬카페 정보 털어오기 완료!!")

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import random
import re
import os

# ==========================================
# 1. 새로운 링크 데이터 불러오기
# ==========================================
file_path = 'bj_links_extracted_full.csv'

try:
    df = pd.read_csv(file_path, encoding='utf-8-sig')
except:
    df = pd.read_csv(file_path, encoding='cp949')

# 팬카페 URL이 있는 스트리머만 골라내기 (None이나 공백 제외)
# 'cafe.naver.com'이 포함된 진짜 링크만 필터링합니다.
target_df = df[df['fancafe_url'].fillna('').str.contains('cafe.naver.com')].copy()

print(f" 총 {len(df)}명 중 팬카페 주소가 확인된 {len(target_df)}명을 대상으로 수집을 시작합니다")

# ==========================================
# 2. 카페 ID 추출 및 불도저 일꾼 설정
# ==========================================
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Referer": "https://www.naver.com/"
}

def extract_cafe_id(url):
    """URL에서 카페 아이디만 쏙 뽑아내는 함수"""
    try:
        # https://cafe.naver.com/아이디 형식에서 아이디만 추출
        clean_url = str(url).split('?')[0].rstrip('/')
        return clean_url.split('/')[-1]
    except:
        return None

def get_cafe_stats_bulldozer(cafe_id):
    """선샌니의 '찐 불도저 방식'  그대로!"""
    url = f"https://cafe.naver.com/{cafe_id}"
    try:
        res = requests.get(url, headers=headers, timeout=10)
        status = res.status_code
        if status == 200:
            soup = BeautifulSoup(res.text, 'html.parser')
            member_count, post_count = "0", "0"
            
            # [가입자 수]
            member_element = soup.select_one('.mem-cnt-info em')
            if member_element:
                member_count = re.sub(r'[^0-9]', '', member_element.text)
                
            # [전체 게시글 수 - 불도저 ]
            all_li_tags = soup.find_all('li')
            for li in all_li_tags:
                text_content = li.text.strip().replace('\n', '').replace(' ', '')
                if '전체글' in text_content:
                    numbers = re.findall(r'[\d,]+', text_content)
                    if numbers:
                        post_count = numbers[0].replace(',', '')
                        break
            return status, member_count, post_count
        else:
            return status, None, None
    except:
        return "ERROR", None, None

# ==========================================
# 3. 자동화 관제탑 가동 (500개씩 끊어서 저장)
# ==========================================
streamers = target_df['streamer_name'].tolist()
cafe_urls = target_df['fancafe_url'].tolist()
total_count = len(streamers)

stop_all = False
chunk_size = 500

print("-" * 50)
print(f" [실시간 관제탑] 직접 링크 기반 불도저 주행을 시작합니다!")

for i in range(0, total_count, chunk_size):
    if stop_all: break
    
    current_chunk_streamers = streamers[i : i + chunk_size]
    current_chunk_urls = cafe_urls[i : i + chunk_size]
    
    start_num = i + 1
    end_num = min(i + chunk_size, total_count)
    file_name = f'fancafe_stats_direct_{start_num}_{end_num}.csv'
    
    if os.path.exists(file_name):
        print(f" {start_num}~{end_num} 묶음은 이미 완료되어 건너뜁니다.")
        continue

    print(f"\n {start_num}번부터 {end_num}번까지 침투 시작!")
    results = []
    
    for j in range(len(current_chunk_streamers)):
        name = current_chunk_streamers[j]
        url = current_chunk_urls[j]
        cafe_id = extract_cafe_id(url)
        
        if not cafe_id:
            print(f" [SKIP] {name} | URL 형식이 이상해요: {url}")
            continue
            
        status, members, posts = get_cafe_stats_bulldozer(cafe_id)
        
        # 선샌니 전용 관제 화면
        status_icon = "✅" if status == 200 else "❌"
        real_num = i + j + 1
        print(f"{status_icon} [{status}] {real_num}/{total_count} | {name}({cafe_id}) ➡️ 멤버: {members} / 글: {posts}")
        
        if status == 403:
            print(f"\n [비상] 403 차단 발생! 중단합니다. 핫스팟 찬스를 써주세요.")
            stop_all = True
            break
            
        results.append({
            'streamer_name': name,
            'cafe_id': cafe_id,
            'fancafe_url': url,
            'member_count': members,
            'post_count': posts
        })
        
        # 지연 시간 (안전하게 4~7초)
        time.sleep(random.uniform(4.0, 7.0))
    
    if results:
        pd.DataFrame(results).to_csv(file_name, index=False, encoding='utf-8-sig')
        if not stop_all:
            print(f" {file_name} 저장 완료! 15초간 엔진을 식힙니다...")
            time.sleep(15)

print("\n 모든 주소 기반 수집 작업이 끝났습니다! 선샌니 고생하셨어요!")